# Run Cosmos T1 Quantized Chat

Chat with the HF 4-bit quantized model on Drive.
Model is loaded once and conversation history is preserved across prompts within the loop.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

candidates = [
    Path('/content/drive/MyDrive/training-embedding'),
    Path('/content/drive/My Drive/training-embedding'),
]
project_root = next((p for p in candidates if p.exists()), None)
if project_root is None:
    raise RuntimeError('training-embedding folder not found under mounted Drive.')

print(f'project_root={project_root}')


In [ ]:
%%bash
set -euo pipefail
python -m pip install -U pip
python -m pip install -U "bitsandbytes>=0.46.1" accelerate transformers sentencepiece


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_DIR = project_root / 'models' / 'turkish-gemma-9b-t1-4bit'
if not MODEL_DIR.exists():
    raise FileNotFoundError(f'Quantized model folder not found: {MODEL_DIR}')

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_DIR),
    device_map='auto',
    dtype=torch.float16,
)

if getattr(model, 'generation_config', None) is not None and hasattr(model.generation_config, 'use_cache'):
    model.generation_config.use_cache = True

print('Loaded quantized model from:', MODEL_DIR)
print('Model device:', model.device)


In [ ]:
# Chat settings
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.2
TOP_P = 0.9
DO_SAMPLE = False
MAX_HISTORY_TURNS = 8

PRE_PROMPT_PREFIX = ''
PRE_PROMPT = ''
PRE_PROMPT_SUFFIX = ''
INPUT_PREFIX = '<start_of_turn>user\n'
INPUT_SUFFIX = '<end_of_turn>\n<start_of_turn>model\n'

print('Chat settings ready.')


In [ ]:
# Interactive chat loop with persistent in-memory history.
# Commands: /reset, /exit

history = []

def build_prompt(history_turns, user_text):
    prompt = f'{PRE_PROMPT_PREFIX}{PRE_PROMPT}{PRE_PROMPT_SUFFIX}'
    for u, a in history_turns[-MAX_HISTORY_TURNS:]:
        prompt += f'{INPUT_PREFIX}{u}{INPUT_SUFFIX}{a}<end_of_turn>\n'
    prompt += f'{INPUT_PREFIX}{user_text}{INPUT_SUFFIX}'
    return prompt

print('Chat started. Type /exit to stop, /reset to clear history.')
while True:
    user_text = input('You: ').strip()
    if not user_text:
        continue
    if user_text in {'/exit', '/quit'}:
        print('Bye.')
        break
    if user_text == '/reset':
        history = []
        print('History cleared.')
        continue

    prompt = build_prompt(history, user_text)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_ids = out[0][inputs['input_ids'].shape[1]:]
    assistant_text = tokenizer.decode(new_ids, skip_special_tokens=True).strip()

    if '<end_of_turn>' in assistant_text:
        assistant_text = assistant_text.split('<end_of_turn>', 1)[0].strip()

    print(f'Assistant: {assistant_text}\n')

    history.append((user_text, assistant_text))
    history = history[-MAX_HISTORY_TURNS:]
